<a href="https://colab.research.google.com/github/dakshag11a/food-delivery-operations-analytics/blob/main/2026b03110270_DakshAgarwal_FoodDeliveryAnalytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Project 8: Food Delivery Operations Analytics
**Industry:** FoodTech / Logistics  
**Student:** Daksh Agarwal | **Student ID:** 2026b03110270  
**Dataset:** `08_food_delivery.csv`  
**Tools:** Python, NumPy, Pandas

**Business scenario:** Evaluate delivery times, order outcomes, restaurant performance and city-level operations to identify service improvements.

In [30]:
import numpy as np
import pandas as pd

**Task 1: Load and inspect the order dataset**

In [31]:
# Read the CSV file into a DataFrame
df = pd.read_csv('08_food_delivery.csv')
print('Shape (rows, columns):', df.shape)
df.head()

Shape (rows, columns): (2000, 10)


,Order_ID,City,Restaurant,Order_Value,Distance_km,Delivery_Time_Min,Order_Status,Payment_Mode,Order_Hour,Record_Source
0,FD00000,Noida,Green Bowl,588.00,2.6,32.0,Delivered,Card,10,Original
1,FD00001,Delhi,Pizza Point,496.68,5.3,36.0,Delivered,Cash,22,Original
2,FD00002,Faridabad,Pizza Point,419.41,6.0,42.0,Delivered,Cash,18,Original
3,FD00003,Noida,Pizza Point,195.48,5.0,40.0,Delivered,UPI,19,Original
4,FD00004,Faridabad,Cafe 24,621.70,2.5,15.0,Delivered,Cash,22,Original


In [32]:
# Column names, data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           2000 non-null   object 
 1   City               2000 non-null   object 
 2   Restaurant         2000 non-null   object 
 3   Order_Value        2000 non-null   float64
 4   Distance_km        2000 non-null   float64
 5   Delivery_Time_Min  1972 non-null   float64
 6   Order_Status       2000 non-null   object 
 7   Payment_Mode       2000 non-null   object 
 8   Order_Hour         2000 non-null   int64  
 9   Record_Source      2000 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 156.4+ KB


In [33]:
# Summary statistics of the number columns
df.describe()

,Order_Value,Distance_km,Delivery_Time_Min,Order_Hour
count,2000.000000,2000.000000,1972.000000,2000.000000
mean,451.894200,5.125550,36.066937,15.526500
std,166.914499,1.931048,11.405162,4.729004
min,80.000000,0.500000,10.000000,8.000000
25%,343.555000,3.800000,28.750000,11.000000
50%,449.880000,5.200000,36.000000,16.000000
75%,559.657500,6.400000,43.000000,20.000000
max,944.270000,10.400000,78.000000,23.000000


In [34]:
# Count of each category in the text columns
for col in ['City', 'Restaurant', 'Order_Status', 'Payment_Mode']:
    print(df[col].value_counts())
    print()

City
Faridabad    585
Delhi        477
Gurugram     470
Noida        468
Name: count, dtype: int64

Restaurant
Pizza Point    452
Green Bowl     426
Urban Bites    402
Spice Hub      380
Cafe 24        340
Name: count, dtype: int64

Order_Status
Delivered    1642
Cancelled     185
Delayed       173
Name: count, dtype: int64

Payment_Mode
Card    731
Cash    690
UPI     579
Name: count, dtype: int64



**Task 2 : Missing delivery times and how to handle them**

In [35]:
# Count missing values in every column
print(df.isnull().sum())

# Which order statuses do the missing rows belong to?
missing = df[df['Delivery_Time_Min'].isnull()]
print()
print('Rows with missing delivery time:', len(missing))
print(missing['Order_Status'].value_counts())

Order_ID              0
City                  0
Restaurant            0
Order_Value           0
Distance_km           0
Delivery_Time_Min    28
Order_Status          0
Payment_Mode          0
Order_Hour            0
Record_Source         0
dtype: int64

Rows with missing delivery time: 28
Order_Status
Delivered    25
Cancelled     2
Delayed       1
Name: count, dtype: int64


**Decision:** Only 28 of 2,000 rows (1.4%) have a missing delivery time. Deleting them would also lose valid order value and status data. Delayed orders take longer than delivered ones, so each missing value is filled with the **median delivery time of its own order status**. The median is used instead of the mean because a few very long deliveries pull the mean up.

In [36]:
# Median delivery time for each order status
status_median = df.groupby('Order_Status')['Delivery_Time_Min'].median()
print(status_median)

# Fill each missing value with the median of its own status
df['Delivery_Time_Min'] = df['Delivery_Time_Min'].fillna(df['Order_Status'].map(status_median))

print()
print('Missing delivery times after filling:', df['Delivery_Time_Min'].isnull().sum())

Order_Status
Cancelled    37.0
Delayed      44.0
Delivered    35.0
Name: Delivery_Time_Min, dtype: float64

Missing delivery times after filling: 0


**Task 3: Check duplicate Order_ID values**

In [37]:
dup_count = df['Order_ID'].duplicated().sum()
print('Duplicate Order_ID values:', dup_count)

if dup_count > 0:
    df = df.drop_duplicates(subset='Order_ID', keep='first')
    print('Duplicates removed. New shape:', df.shape)
else:
    print('No duplicates found, so no rows were removed.')

Duplicate Order_ID values: 0
No duplicates found, so no rows were removed.


In [38]:
output_file = 'cleaned_food_delivery.csv'
df.to_csv(output_file, index=False)
print('Cleaned file saved!')

Cleaned file saved!


**Task 4: Total orders and order-status percentages**

In [39]:
total_orders = len(df)
print('Total orders:', total_orders)

status_counts = df['Order_Status'].value_counts()
status_pct = (df['Order_Status'].value_counts(normalize=True) * 100).round(2)

status_summary = pd.DataFrame({'Orders': status_counts, 'Percentage': status_pct})
status_summary

Total orders: 2000


,Orders,Percentage
Order_Status,,
Delivered,1642,82.10
Cancelled,185,9.25
Delayed,173,8.65


**Task 5: Average delivery time by city**

In [40]:
avg_time_city = df.groupby('City')['Delivery_Time_Min'].mean().round(2).sort_values()
avg_time_city

,Delivery_Time_Min
City,
Delhi,35.65
Noida,36.00
Faridabad,36.05
Gurugram,36.54


**Task 6: Five restaurants with highest total order value**


The dataset has only five restaurants, so this ranks all of them.

In [41]:
restaurant_total = df.groupby('Restaurant')['Order_Value'].sum().round(2)
top5 = restaurant_total.sort_values(ascending=False).head(5)
top5

,Order_Value
Restaurant,
Pizza Point,202159.89
Green Bowl,189623.29
Urban Bites,181800.76
Spice Hub,173260.41
Cafe 24,156944.05


**Task 7: Average order value by restaurant**

In [42]:
restaurant_avg = df.groupby('Restaurant')['Order_Value'].mean().round(2).sort_values(ascending=False)
restaurant_avg

,Order_Value
Restaurant,
Cafe 24,461.60
Spice Hub,455.95
Urban Bites,452.24
Pizza Point,447.26
Green Bowl,445.13


In [43]:
# Restaurant ranking table (deliverable)
restaurant_rank = df.groupby('Restaurant').agg(
    Orders=('Order_ID', 'count'),
    Total_Value=('Order_Value', 'sum'),
    Avg_Value=('Order_Value', 'mean'),
    Avg_Delivery_Min=('Delivery_Time_Min', 'mean')
).round(2)

restaurant_rank = restaurant_rank.sort_values('Total_Value', ascending=False)
restaurant_rank['Rank'] = range(1, len(restaurant_rank) + 1)
restaurant_rank

,Orders,Total_Value,Avg_Value,Avg_Delivery_Min,Rank
Restaurant,,,,,
Pizza Point,452,202159.89,447.26,36.39,1
Green Bowl,426,189623.29,445.13,36.46,2
Urban Bites,402,181800.76,452.24,36.42,3
Spice Hub,380,173260.41,455.95,35.82,4
Cafe 24,340,156944.05,461.60,34.95,5


**Task 8: Identify peak ordering hours**

In [44]:
# Number of orders in each hour
hourly = df['Order_Hour'].value_counts().sort_index()

# Top 5 busiest hours with their share of all orders
peak_hours = hourly.sort_values(ascending=False).head(5)
peak_summary = pd.DataFrame({
    'Orders': peak_hours,
    'Share_%': (peak_hours / total_orders * 100).round(2)
})
print('Busiest hour:', hourly.idxmax(), ':00 with', hourly.max(), 'orders')
peak_summary

Busiest hour: 10 :00 with 161 orders


,Orders,Share_%
Order_Hour,,
10,161,8.05
13,153,7.65
21,148,7.40
18,147,7.35
9,136,6.80


**Task 9: Compare delivery time across order-status categories**

In [45]:
status_time = df.groupby('Order_Status')['Delivery_Time_Min'].agg(['count', 'mean', 'median', 'min', 'max']).round(2)
status_time

,count,mean,median,min,max
Order_Status,,,,,
Cancelled,185,36.31,37.0,10.0,74.0
Delayed,173,42.74,44.0,13.0,78.0
Delivered,1642,35.33,35.0,10.0,78.0


**Task 10: Cancellation and delay rates by city**

In [46]:
# Count orders by city and status
rates = pd.pivot_table(df, index='City', columns='Order_Status',
                       values='Order_ID', aggfunc='count', fill_value=0)

rates['Total'] = rates['Cancelled'] + rates['Delayed'] + rates['Delivered']
rates['Cancel_Rate_%'] = (rates['Cancelled'] / rates['Total'] * 100).round(2)
rates['Delay_Rate_%'] = (rates['Delayed'] / rates['Total'] * 100).round(2)
rates.sort_values('Cancel_Rate_%', ascending=False)

Order_Status,Cancelled,Delayed,Delivered,Total,Cancel_Rate_%,Delay_Rate_%
City,,,,,,
Delhi,53,44,380,477,11.11,9.22
Gurugram,44,38,388,470,9.36,8.09
Noida,40,39,389,468,8.55,8.33
Faridabad,48,52,485,585,8.21,8.89


**Task 11: Median and standard deviation of delivery time (NumPy)**

In [47]:
times = df['Delivery_Time_Min'].to_numpy()   # convert the column to a NumPy array

print('Median delivery time:', np.median(times), 'min')
print('Standard deviation   :', round(np.std(times), 2), 'min')
print('Mean delivery time    :', round(np.mean(times), 2), 'min')

Median delivery time: 36.0 min
Standard deviation   : 11.32 min
Mean delivery time    : 36.06 min


**Task 12: Unusually long deliveries using IQR**
A delivery is unusually long if it is above **Q3 + 1.5 x IQR**.

In [48]:
def iqr_upper_fence(values):
    # Return Q1, Q3, IQR and the upper fence for a NumPy array
    q1 = np.percentile(values, 25)
    q3 = np.percentile(values, 75)
    iqr = q3 - q1
    return q1, q3, iqr, q3 + 1.5 * iqr

q1, q3, iqr, upper = iqr_upper_fence(times)
print('Q1:', q1, '| Q3:', q3, '| IQR:', iqr)
print('Upper fence:', upper, 'minutes')

long_orders = df[df['Delivery_Time_Min'] > upper]
print('Unusually long deliveries:', len(long_orders))
print('Share of all orders:', round(len(long_orders) / len(df) * 100, 2), '%')

Q1: 29.0 | Q3: 43.0 | IQR: 14.0
Upper fence: 64.0 minutes
Unusually long deliveries: 15
Share of all orders: 0.75 %


In [49]:
print(long_orders['City'].value_counts())
print()
print(long_orders['Order_Status'].value_counts())

long_orders.sort_values('Delivery_Time_Min', ascending=False).head(10)

City
Faridabad    6
Gurugram     4
Noida        3
Delhi        2
Name: count, dtype: int64

Order_Status
Delivered    10
Delayed       4
Cancelled     1
Name: count, dtype: int64


,Order_ID,City,Restaurant,Order_Value,Distance_km,Delivery_Time_Min,Order_Status,Payment_Mode,Order_Hour,Record_Source
701,FD00701,Faridabad,Green Bowl,671.62,4.2,78.0,Delivered,Cash,9,Synthetic
651,FD00651,Faridabad,Green Bowl,256.51,2.9,78.0,Delivered,Card,19,Synthetic
1901,FD01901,Faridabad,Spice Hub,488.38,5.9,78.0,Delayed,Cash,17,Synthetic
1031,FD01031,Gurugram,Urban Bites,691.11,4.4,78.0,Delivered,UPI,16,Synthetic
723,FD00723,Noida,Pizza Point,392.78,5.7,75.0,Delayed,Card,14,Synthetic
454,FD00454,Noida,Spice Hub,497.76,6.9,75.0,Delivered,UPI,20,Original
718,FD00718,Delhi,Cafe 24,283.90,2.4,74.0,Cancelled,UPI,15,Synthetic
825,FD00825,Delhi,Urban Bites,629.15,6.7,72.0,Delivered,UPI,10,Synthetic
604,FD00604,Gurugram,Pizza Point,250.84,5.1,72.0,Delayed,Card,20,Synthetic
57,FD00057,Faridabad,Spice Hub,479.34,3.5,71.0,Delivered,Card,10,Original


**Task 13: Delivery-time bands (Fast, Standard, Delayed)**
- **Fast:** 30 minutes or less
- **Standard:** 31 to 45 minutes
- **Delayed:** more than 45 minutes

In [50]:
conditions = [df['Delivery_Time_Min'] <= 30,
              df['Delivery_Time_Min'] <= 45]
choices = ['Fast', 'Standard']

df['Delivery_Band'] = np.select(conditions, choices, default='Delayed')

band_summary = pd.DataFrame({
    'Orders': df['Delivery_Band'].value_counts(),
    'Percentage': (df['Delivery_Band'].value_counts(normalize=True) * 100).round(2)
})
band_summary.loc[['Fast', 'Standard', 'Delayed']]

,Orders,Percentage
Delivery_Band,,
Fast,624,31.2
Standard,984,49.2
Delayed,392,19.6


In [51]:
# Share of Delayed-band orders in each city
df['Is_Delayed_Band'] = df['Delivery_Band'] == 'Delayed'
(df.groupby('City')['Is_Delayed_Band'].mean() * 100).round(2).sort_values(ascending=False)

,Is_Delayed_Band
City,
Noida,21.15
Gurugram,20.85
Faridabad,19.83
Delhi,16.56


**Task 14: Group revenue by city and payment mode**
Revenue is the total order value of **Delivered** orders, since cancelled orders do not produce revenue. The all-orders version is shown after it.

In [52]:
delivered = df[df['Order_Status'] == 'Delivered']

revenue = pd.pivot_table(delivered, index='City', columns='Payment_Mode',
                         values='Order_Value', aggfunc='sum',
                         margins=True, margins_name='Total').round(2)
revenue

Payment_Mode,Card,Cash,UPI,Total
City,,,,
Delhi,63130.15,54907.67,50234.89,168272.71
Faridabad,85471.12,76910.83,53791.13,216173.08
Gurugram,62048.31,60139.90,58180.78,180368.99
Noida,60119.38,66678.76,51817.95,178616.09
Total,270768.96,258637.16,214024.75,743430.87


In [53]:
# Same grouping for all orders (for comparison)
df.groupby(['City', 'Payment_Mode'])['Order_Value'].sum().round(2).unstack()

Payment_Mode,Card,Cash,UPI
City,,,
Delhi,83330.42,67419.72,63846.83
Faridabad,101539.15,93535.09,65497.19
Gurugram,76164.07,73573.77,68255.97
Noida,70018.18,76911.12,63696.89


## Deliverable: City performance table

In [54]:
city_perf = df.groupby('City').agg(
    Orders=('Order_ID', 'count'),
    Avg_Delivery_Min=('Delivery_Time_Min', 'mean'),
    Avg_Order_Value=('Order_Value', 'mean')
).round(2)

city_perf['Cancel_Rate_%'] = rates['Cancel_Rate_%']
city_perf['Delay_Rate_%'] = rates['Delay_Rate_%']
city_perf['Delivered_Revenue'] = delivered.groupby('City')['Order_Value'].sum().round(2)
city_perf.sort_values('Delivered_Revenue', ascending=False)

,Orders,Avg_Delivery_Min,Avg_Order_Value,Cancel_Rate_%,Delay_Rate_%,Delivered_Revenue
City,,,,,,
Faridabad,585,36.05,445.42,8.21,8.89,216173.08
Gurugram,470,36.54,463.82,9.36,8.09,180368.99
Noida,468,36.00,450.06,8.55,8.33,178616.09
Delhi,477,35.65,449.89,11.11,9.22,168272.71


## Extra analysis: what else affects service quality?

In [55]:
# 1. Does distance affect delivery time?
conditions = [df['Distance_km'] <= 3, df['Distance_km'] <= 6, df['Distance_km'] <= 9]
choices = ['0-3 km', '3-6 km', '6-9 km']
df['Distance_Band'] = np.select(conditions, choices, default='9+ km')

df.groupby('Distance_Band')['Delivery_Time_Min'].agg(['count', 'mean']).round(2).sort_values('mean')

,count,mean
Distance_Band,,
0-3 km,312,31.35
3-6 km,1036,35.73
6-9 km,611,38.55
9+ km,41,42.98


In [56]:
# 2. Which hours have the slowest deliveries?
df.groupby('Order_Hour')['Delivery_Time_Min'].mean().round(2).sort_values(ascending=False).head(5)

,Delivery_Time_Min
Order_Hour,
14,39.48
12,39.02
20,38.50
19,38.48
13,38.04


In [57]:
# 3. Share of problem orders (Cancelled or Delayed) by hour and by restaurant
df['Is_Problem'] = df['Order_Status'] != 'Delivered'

print('By hour (top 5):')
print((df.groupby('Order_Hour')['Is_Problem'].mean() * 100).round(2).sort_values(ascending=False).head(5))
print()
print('By restaurant:')
print((df.groupby('Restaurant')['Is_Problem'].mean() * 100).round(2).sort_values(ascending=False))

By hour (top 5):
Order_Hour
8     21.97
21    21.62
12    21.21
13    20.26
20    20.00
Name: Is_Problem, dtype: float64

By restaurant:
Restaurant
Pizza Point    19.47
Urban Bites    18.66
Spice Hub      18.42
Green Bowl     17.61
Cafe 24        14.71
Name: Is_Problem, dtype: float64


## Testing the results
Simple checks that the numbers add up. If any check fails, Python stops with an AssertionError.

In [58]:
assert df['Delivery_Time_Min'].isnull().sum() == 0, 'Missing delivery times remain'
assert df['Order_ID'].is_unique, 'Duplicate Order_ID found'
assert status_counts.sum() == total_orders, 'Status counts do not add up'
assert rates['Total'].sum() == total_orders, 'City totals do not add up'
assert band_summary['Orders'].sum() == total_orders, 'Band counts do not add up'
assert round(delivered['Order_Value'].sum(), 2) == revenue.loc['Total', 'Total'], 'Revenue does not match'
print('All checks passed.')

All checks passed.


**Task 15: Five operational recommendations**

1. **Reduce cancellations in Delhi.** Delhi has the highest cancellation rate (11.11%) against 8.21% to 9.36% in the other cities. Review rider availability and restaurant acceptance times in Delhi first; bringing it to the 8.5% level of the other cities would recover about 12 orders per 477.
2. **Plan rider capacity around the four peak hours.** Orders peak at 10:00, 13:00, 18:00 and 21:00 (7.4% to 8.1% of orders each). The slowest average delivery times are at 12:00 to 14:00 (38 to 39.5 minutes) and the highest problem-order rates at 08:00, 21:00 and 12:00 (above 21%). Schedule extra riders in these windows.
3. **Control long-distance orders.** Average delivery time rises from 31.4 minutes (0 to 3 km) to 38.6 minutes (6 to 9 km) and 43.0 minutes (9+ km). Limit the delivery radius during peak hours or assign nearer riders to orders above 6 km.
4. **Learn from Cafe 24 and fix Pizza Point.** Cafe 24 has the lowest problem-order rate (14.71%) and fastest average delivery (34.95 min). Pizza Point earns the most revenue but has the highest problem rate (19.47%). Share Cafe 24's preparation and handover practice with the other restaurants.
5. **Track outliers and improve data capture.** 15 deliveries (0.75%) exceed the IQR upper fence of 64 minutes, six of them in Faridabad. Also, 19.6% of orders fall in the Delayed band (over 45 minutes) but only 8.65% are labelled Delayed, so the status label and the time threshold are not aligned. Define one clear delay rule and record all delivery times (28 were missing).
